# DAA Lab 1: Comparative Study of Algorithm Efficiency and Performance Analysis

**Student:** Pratyush Jha  
**Roll Number:** 2401201017  
**Course:** BCA (AI & Data Science), Section B  
**Course Code:** ENCA301  

This notebook contains the implementations and experimental-analysis code for the Lab 1 study:
- Bubble Sort
- Insertion Sort
- Merge Sort
- Quick Sort
- Fibonacci: Recursive, Iterative, Dynamic Programming
- Execution-time, comparison-count and memory measurements
- Visualization code

In [ ]:
import random
import time
import tracemalloc
import math
import pandas as pd
import matplotlib.pyplot as plt

SIZES = [100, 500, 1000, 5000, 10000]
FIB_N = [10, 20, 30, 40]
RUNS = 5

def median(values):
    return sorted(values)[len(values) // 2]


## 1. Sorting Algorithms

In [ ]:
def bubble_sort(arr):
    a = arr.copy()
    comparisons = 0

    for i in range(len(a) - 1):
        swapped = False
        for j in range(len(a) - 1 - i):
            comparisons += 1
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                swapped = True
        if not swapped:
            break

    return a, comparisons


def insertion_sort(arr):
    a = arr.copy()
    comparisons = 0

    for i in range(1, len(a)):
        key = a[i]
        j = i - 1

        while j >= 0:
            comparisons += 1
            if a[j] <= key:
                break
            a[j + 1] = a[j]
            j -= 1

        a[j + 1] = key

    return a, comparisons


In [ ]:
def merge_sort(arr):
    comparisons = 0

    def merge(left, right):
        nonlocal comparisons
        result = []
        i = j = 0

        while i < len(left) and j < len(right):
            comparisons += 1
            if left[i] <= right[j]:
                result.append(left[i])
                i += 1
            else:
                result.append(right[j])
                j += 1

        result.extend(left[i:])
        result.extend(right[j:])
        return result

    def sort(a):
        if len(a) <= 1:
            return a
        mid = len(a) // 2
        return merge(sort(a[:mid]), sort(a[mid:]))

    return sort(arr.copy()), comparisons


def quick_sort(arr):
    a = arr.copy()
    comparisons = 0

    def partition(low, high):
        nonlocal comparisons
        pivot_index = random.randint(low, high)
        a[pivot_index], a[high] = a[high], a[pivot_index]
        pivot = a[high]
        i = low

        for j in range(low, high):
            comparisons += 1
            if a[j] <= pivot:
                a[i], a[j] = a[j], a[i]
                i += 1

        a[i], a[high] = a[high], a[i]
        return i

    def sort(low, high):
        if low < high:
            p = partition(low, high)
            sort(low, p - 1)
            sort(p + 1, high)

    sort(0, len(a) - 1)
    return a, comparisons


## 2. Fibonacci Algorithms

In [ ]:
def fib_recursive(n):
    if n <= 1:
        return n
    return fib_recursive(n - 1) + fib_recursive(n - 2)


def fib_iterative(n):
    if n <= 1:
        return n

    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b


def fib_dp(n):
    memo = {}

    def solve(k):
        if k in memo:
            return memo[k]
        if k <= 1:
            return k
        memo[k] = solve(k - 1) + solve(k - 2)
        return memo[k]

    return solve(n)


## 3. Correctness Check

In [ ]:
sample = [8, 3, 5, 1, 9, 2]

for name, fn in [
    ("Bubble Sort", bubble_sort),
    ("Insertion Sort", insertion_sort),
    ("Merge Sort", merge_sort),
    ("Quick Sort", quick_sort),
]:
    result, comparisons = fn(sample)
    print(f"{name:15} -> {result} | comparisons: {comparisons}")

print("\nFibonacci:")
for n in [10, 20, 30, 40]:
    print(n, fib_recursive(n), fib_iterative(n), fib_dp(n))


## 4. Experimental Measurement

In [ ]:
SORTS = {
    "Bubble Sort": bubble_sort,
    "Insertion Sort": insertion_sort,
    "Merge Sort": merge_sort,
    "Quick Sort": quick_sort,
}

def make_input(n, kind):
    if kind == "Sorted":
        return list(range(n))
    if kind == "Reverse":
        return list(range(n, 0, -1))
    if kind == "Random":
        return random.sample(range(0, max(2 * n, 100000)), n)
    raise ValueError("Unknown input type")


def measure_sort(fn, data, runs=RUNS):
    times = []
    memories = []
    comparisons = []

    for _ in range(runs):
        tracemalloc.start()
        start = time.perf_counter()
        _, c = fn(data)
        elapsed = time.perf_counter() - start
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        times.append(elapsed)
        memories.append(peak / 1024)
        comparisons.append(c)

    return median(times), median(memories), median(comparisons)


results = []

for algorithm, fn in SORTS.items():
    for kind in ["Sorted", "Reverse", "Random"]:
        for n in SIZES:
            data = make_input(n, kind)
            t, mem, comp = measure_sort(fn, data)
            results.append({
                "Algorithm": algorithm,
                "Type": kind,
                "n": n,
                "Time (s)": t,
                "Memory (KB)": mem,
                "Comparisons": comp
            })

sorting_results = pd.DataFrame(results)
sorting_results.head()


## 5. Fibonacci Measurement

In [ ]:
FIBS = {
    "Recursive": fib_recursive,
    "Iterative": fib_iterative,
    "Dynamic Programming": fib_dp,
}

fib_results = []

for algorithm, fn in FIBS.items():
    for n in FIB_N:
        times = []
        memories = []

        for _ in range(RUNS):
            tracemalloc.start()
            start = time.perf_counter()
            value = fn(n)
            elapsed = time.perf_counter() - start
            _, peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            times.append(elapsed)
            memories.append(peak / 1024)

        fib_results.append({
            "Algorithm": algorithm,
            "n": n,
            "Result": value,
            "Time (s)": median(times),
            "Memory (KB)": median(memories)
        })

fib_results = pd.DataFrame(fib_results)
fib_results


## 6. Visualizations

In [ ]:
# Execution time by input type
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=False)

for ax, kind in zip(axes, ["Sorted", "Reverse", "Random"]):
    subset = sorting_results[sorting_results["Type"] == kind]
    for algorithm in SORTS:
        d = subset[subset["Algorithm"] == algorithm]
        ax.plot(d["n"], d["Time (s)"], marker="o", label=algorithm)
    ax.set_yscale("log")
    ax.set_title(f"Input: {kind}")
    ax.set_xlabel("Input Size (n)")
    ax.set_ylabel("Execution Time (seconds)")
    ax.grid(True, which="both", alpha=0.25)

axes[-1].legend()
plt.suptitle("Sorting Algorithm Execution Time vs Input Size (Log Scale)")
plt.tight_layout()
plt.show()


In [ ]:
# Memory usage by input type
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, kind in zip(axes, ["Sorted", "Reverse", "Random"]):
    subset = sorting_results[sorting_results["Type"] == kind]
    for algorithm in SORTS:
        d = subset[subset["Algorithm"] == algorithm]
        ax.plot(d["n"], d["Memory (KB)"], marker="s", label=algorithm)
    ax.set_yscale("log")
    ax.set_title(f"Input: {kind}")
    ax.set_xlabel("Input Size (n)")
    ax.set_ylabel("Peak Memory (KB)")
    ax.grid(True, which="both", alpha=0.25)

axes[-1].legend()
plt.suptitle("Sorting Algorithm Memory Usage vs Input Size (Log Scale)")
plt.tight_layout()
plt.show()


In [ ]:
# Comparison counts
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, kind in zip(axes, ["Sorted", "Reverse", "Random"]):
    subset = sorting_results[sorting_results["Type"] == kind]
    for algorithm in SORTS:
        d = subset[subset["Algorithm"] == algorithm]
        ax.plot(d["n"], d["Comparisons"], marker="^", label=algorithm)
    ax.set_title(f"Input: {kind}")
    ax.set_xlabel("Input Size (n)")
    ax.set_ylabel("Number of Comparisons")
    ax.grid(True, alpha=0.25)

axes[-1].legend()
plt.suptitle("Number of Comparisons vs Input Size")
plt.tight_layout()
plt.show()


In [ ]:
# Fibonacci execution time and memory
fig, ax = plt.subplots(figsize=(9, 5))
for algorithm in FIBS:
    d = fib_results[fib_results["Algorithm"] == algorithm]
    ax.plot(d["n"], d["Time (s)"], marker="o", label=algorithm)
ax.set_xlabel("n (Fibonacci Index)")
ax.set_ylabel("Execution Time (seconds)")
ax.set_title("Fibonacci Algorithm Execution Time vs n")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
for algorithm in FIBS:
    d = fib_results[fib_results["Algorithm"] == algorithm]
    ax.plot(d["n"], d["Memory (KB)"], marker="s", label=algorithm)
ax.set_xlabel("n (Fibonacci Index)")
ax.set_ylabel("Peak Memory (KB)")
ax.set_title("Fibonacci Algorithm Memory Usage vs n")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## 7. Complexity Summary

| Algorithm | Best Case | Average Case | Worst Case | Space |
|---|---|---|---|---|
| Bubble Sort | O(n) | O(n²) | O(n²) | O(1) |
| Insertion Sort | O(n) | O(n²) | O(n²) | O(1) |
| Merge Sort | O(n log n) | O(n log n) | O(n log n) | O(n) |
| Quick Sort | O(n log n) | O(n log n) | O(n²) | O(log n) |
| Fibonacci (Recursive) | O(2ⁿ) | O(2ⁿ) | O(2ⁿ) | O(n) |
| Fibonacci (Iterative) | O(n) | O(n) | O(n) | O(1) |
| Fibonacci (Dynamic Programming) | O(n) | O(n) | O(n) | O(n) |

The measured results in the report were obtained using the same experimental structure: five runs with the median used for timing and memory measurements.